# 01. Merge, decontaminate and sample the training pool

Loads the source datasets, merges them, removes training instances that overlap with any held-out set (word n-gram overlap on claims and on evidence) and draws the label-balanced sample of 3,582 claims.

Inputs: `../data/raw/train/` and `../data/raw/test/` (see `../data/README.md`).

Output: `../data/training/sample.jsonl` (columns `claim`, `evidence`, `label`, `dataset`).


#### Parameters

In [ ]:
# Parameters
OUTPUT_DIR = "../data/training/"

# Train
TRAIN_DIR = "../data/raw/train/"
TRAIN_PUBMEDQA_LABELED_FILE_PATH = "ori_pqal.json"
TRAIN_PUBMEDQA_ARTIFICIAL_FILE_PATH = "train-00000-of-00001.parquet"
TRAIN_BIOASQ_FILE_PATH = "training12b_new.json"
TRAIN_COVIDFACT_FILE_PATH = "COVIDFACT_dataset.jsonl"
TRAIN_HEALTHVER_FILE_PATH = "Healthver_train.json"
TRAIN_FEVEROUS_FILE_PATH = "Feverous_train.json"
TRAIN_HOVER_FILE_PATH = "Hover_train.json"

# Test
TEST_DIR = "../data/raw/test/"
TEST_HEALTHFC_FILE_PATH = "../data/healthfc/healthFC_annotated_tavily_final.csv"  # versioned evaluation file
TEST_HEALTHFC_RAW_FILE_PATH = "healthFC_annotated.csv"
TEST_SCIFACT_NO_NEI_FILE_PATH = "scifact_no-nei_dataset.csv"
TEST_COVERT_FC_FILE_PATH = "CoVERT_FC_annotations.jsonl"
TEST_BIOASQ_FILE_PATH_1 = "12B1_golden.json"
TEST_BIOASQ_FILE_PATH_2 = "12B2_golden.json"
TEST_BIOASQ_FILE_PATH_3 = "12B3_golden.json"
TEST_BIOASQ_FILE_PATH_4 = "12B4_golden.json"
TEST_HEALTHVER_FILE_PATH = "Healthver_test.json"
TEST_FEVEROUS_FILE_PATH = "FEVEROUS.json"
TEST_HOVER_FILE_PATH = "HOVER.json"
TEST_LLM_AGGREFACT_FILE_PATH = "LLM-AggreFact_test.json"
TEST_FEVEROUS_OPEN_FILE_PATH = "Open_FEVEROUS.json"
TEST_HEALTHVER_OPEN_FILE_PATH = "Open_Healthver_test.json"
TEST_HOVER_OPEN_FILE_PATH = "Open_HOVER.json"
TEST_SCIFACT_DEV_FILE_PATH = "Scifact_dev.json"
TEST_SCIFACT_TRAIN_FILE_PATH = "Scifact_train.json"
TEST_VITAMINC_DEV_FILE_PATH = "VitaminC_dev.json"
TEST_VITAMINC_TEST_FILE_PATH = "VitaminC_test.json"

#### Load libraries

In [ ]:
import os
import pandas as pd
import json

In [ ]:
train_mapping = {
    'pubmedqa': os.path.join(TRAIN_DIR, TRAIN_PUBMEDQA_LABELED_FILE_PATH),
    'bioasq': os.path.join(TRAIN_DIR, TRAIN_BIOASQ_FILE_PATH),
    'pubmedqa_artificial': os.path.join(TRAIN_DIR, TRAIN_PUBMEDQA_ARTIFICIAL_FILE_PATH),
    'covidfact': os.path.join(TRAIN_DIR, TRAIN_COVIDFACT_FILE_PATH),
    'feverous': os.path.join(TRAIN_DIR, TRAIN_FEVEROUS_FILE_PATH),
    'healthver': os.path.join(TRAIN_DIR, TRAIN_HEALTHVER_FILE_PATH),
    'hover': os.path.join(TRAIN_DIR, TRAIN_HOVER_FILE_PATH),
}

test_mapping = {
    'bioasq_12b1': os.path.join(TEST_DIR, TEST_BIOASQ_FILE_PATH_1),
    'bioasq_12b2': os.path.join(TEST_DIR, TEST_BIOASQ_FILE_PATH_2),
    'bioasq_12b3': os.path.join(TEST_DIR, TEST_BIOASQ_FILE_PATH_3),
    'bioasq_12b4': os.path.join(TEST_DIR, TEST_BIOASQ_FILE_PATH_4),
    'covert_fc': os.path.join(TEST_DIR, TEST_COVERT_FC_FILE_PATH),
    'feverous': os.path.join(TEST_DIR, TEST_FEVEROUS_FILE_PATH),
    'healthfc': TEST_HEALTHFC_FILE_PATH,
    'healthfc_raw': os.path.join(TEST_DIR, TEST_HEALTHFC_RAW_FILE_PATH),
    'healthver': os.path.join(TEST_DIR, TEST_HEALTHVER_FILE_PATH),
    'hover': os.path.join(TEST_DIR, TEST_HOVER_FILE_PATH),
    'llm_aggrefact': os.path.join(TEST_DIR, TEST_LLM_AGGREFACT_FILE_PATH),
    'feverous_open': os.path.join(TEST_DIR, TEST_FEVEROUS_OPEN_FILE_PATH),
    'healthver_open': os.path.join(TEST_DIR, TEST_HEALTHVER_OPEN_FILE_PATH),
    'hover_open': os.path.join(TEST_DIR, TEST_HOVER_OPEN_FILE_PATH),
    'scifact_dev': os.path.join(TEST_DIR, TEST_SCIFACT_DEV_FILE_PATH),
    'scifact_train': os.path.join(TEST_DIR, TEST_SCIFACT_TRAIN_FILE_PATH),
    'scifact_no_nei': os.path.join(TEST_DIR, TEST_SCIFACT_NO_NEI_FILE_PATH),
    'vitaminc_dev': os.path.join(TEST_DIR, TEST_VITAMINC_DEV_FILE_PATH),
    'vitaminc_test': os.path.join(TEST_DIR, TEST_VITAMINC_TEST_FILE_PATH)
}

#### Train

##### Covid-Fact

In [ ]:
df_covid_fact = pd.read_json(train_mapping['covidfact'], lines=True)
print(df_covid_fact.shape)
print(df_covid_fact.label.value_counts(dropna=False))
df_covid_fact.head()

In [ ]:
df_covid_fact = df_covid_fact[["claim", "evidence", "label"]]
df_covid_fact["dataset"] = "covidfact"
df_covid_fact["label"] = df_covid_fact["label"].apply(lambda x: "supports" if x == "SUPPORTED" else "refutes")
print(df_covid_fact.label.value_counts(dropna=False))
df_covid_fact.head()

##### Feverous

In [ ]:
df_feverous = pd.read_json(train_mapping['feverous'])
print(df_feverous.shape)
print(df_feverous.label.value_counts(dropna=False))
df_feverous.head()

In [ ]:
df_feverous["dataset"] = "feverous"
df_feverous.head()

##### HealthVer

In [ ]:
df_healthver = pd.read_json(train_mapping['healthver'])
print(df_healthver.shape)
print(df_healthver.label.value_counts(dropna=False))
df_healthver.head()

In [ ]:
df_healthver["dataset"] = "healthver"
df_healthver["label"] = df_healthver["label"].apply(lambda x: "supports" if x == "Supports" else ("refutes" if x == "Refutes" else "neutral"))

print(df_healthver.label.value_counts(dropna=False))
df_healthver.head()

##### Hover

In [ ]:
df_hover = pd.read_json(train_mapping['hover'])
print(df_hover.shape)
print(df_hover.label.value_counts(dropna=False))
df_hover.head()

In [ ]:
df_hover["dataset"] = "hover"
df_hover.head()

##### PubMedQA Labeled

In [ ]:
df_pubmedqa = pd.read_json(train_mapping['pubmedqa'], orient="index")
print(df_pubmedqa.shape)
print(df_pubmedqa.final_decision.value_counts(dropna=False))
df_pubmedqa.head()

In [ ]:
df_pubmedqa["dataset"] = "pubmedqa"
df_pubmedqa.final_decision = df_pubmedqa.final_decision.apply(lambda x: "supports" if x == "yes" else ("refutes" if x == "no" else "maybe"))
df_pubmedqa = df_pubmedqa.rename(columns={"QUESTION": "claim", "CONTEXTS": "evidence", "final_decision": "label"})
df_pubmedqa = df_pubmedqa[["claim", "evidence", "label", "dataset"]]
print(df_pubmedqa.label.value_counts(dropna=False))
df_pubmedqa.head()

##### PubMedQA Artificial

In [ ]:
df_pubmedqa_artificial = pd.read_parquet(train_mapping['pubmedqa_artificial'])
print(df_pubmedqa_artificial.shape)
print(df_pubmedqa_artificial.final_decision.value_counts(dropna=False))
df_pubmedqa_artificial.head()

In [ ]:
def sample_balanced(df, label_col="final_decision", n_samples=2000, random_state=42):
    # Get unique labels
    labels = df[label_col].unique()
    print(f"Unique labels: {labels}")
    samples_per_label = n_samples // len(labels)
    print(f"Samples per label: {samples_per_label}")
    # Sample for each label
    sampled = [df[df[label_col] == label].sample(samples_per_label, random_state=random_state) for label in labels]
    # Concatenate and shuffle
    result = pd.concat(sampled).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return result

df_pubmedqa_artificial = sample_balanced(df_pubmedqa_artificial, label_col="final_decision", n_samples=2000)
df_pubmedqa_artificial["final_decision"].value_counts()

In [ ]:
df_pubmedqa_artificial["dataset"] = "pubmedqa_artificial"
df_pubmedqa_artificial.final_decision = df_pubmedqa_artificial.final_decision.apply(lambda x: "supports" if x == "yes" else "refutes")
df_pubmedqa_artificial = df_pubmedqa_artificial.rename(columns={"question": "claim", "context": "evidence", "final_decision": "label"})
df_pubmedqa_artificial = df_pubmedqa_artificial[["claim", "evidence", "label", "dataset"]]
print(df_pubmedqa_artificial.label.value_counts(dropna=False))
df_pubmedqa_artificial.head()

##### Bio Asq

In [ ]:
def load_bioasq_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    if 'questions' in data:
        return pd.DataFrame(data['questions'])
    else:
        return pd.DataFrame(data)

df_bioasq = load_bioasq_data(train_mapping['bioasq'])
print(df_bioasq.shape)
df_bioasq.head()

In [ ]:
df_bioasq.type.value_counts(dropna=False)

In [ ]:
df_bioasq["dataset"] = "bioasq"
df_bioasq = df_bioasq[df_bioasq["type"] == "yesno"]
print(df_bioasq.exact_answer.value_counts(dropna=False))

In [ ]:
df_bioasq = df_bioasq.rename(columns={"body": "claim", "snippets": "evidence", "exact_answer": "label"})
df_bioasq = df_bioasq[["claim", "evidence", "label", "dataset"]]
print(df_bioasq.label.value_counts(dropna=False))
df_bioasq.head()


In [ ]:
df_bioasq.label = df_bioasq.label.apply(lambda x: "supports" if x == "yes" else "refutes")
print(df_bioasq.label.value_counts(dropna=False))
df_bioasq.head()

In [ ]:
def parse_evidence(evidence_list):
    """
    Parse evidence list of dicts to a string with text and link separated by newline.
    """
    parsed = []
    for item in evidence_list:
        text = item.get('text', '')
        link = item.get('document', '')
        if text and link:
            parsed.append(f"{text}\n{link}")
        elif text:
            parsed.append(text)
        elif link:
            parsed.append(link)
    return "\n\n".join(parsed)

# Apply to the evidence column
df_bioasq['evidence'] = df_bioasq['evidence'].apply(parse_evidence)

In [ ]:
df_bioasq.head()

#### Test

##### BioAsq

In [ ]:
df_bioasq_12b1 = load_bioasq_data(test_mapping['bioasq_12b1'])
print(df_bioasq_12b1.shape)
df_bioasq_12b1.head()

In [ ]:
df_bioasq_12b2 = load_bioasq_data(test_mapping['bioasq_12b2'])
print(df_bioasq_12b2.shape)
df_bioasq_12b2.head()

In [ ]:
df_bioasq_12b3 = load_bioasq_data(test_mapping['bioasq_12b3'])
print(df_bioasq_12b3.shape)
df_bioasq_12b3.head()

In [ ]:
df_bioasq_12b4 = load_bioasq_data(test_mapping['bioasq_12b4'])
print(df_bioasq_12b4.shape)
df_bioasq_12b4.head()

In [ ]:
df_bioasq_test = pd.concat([df_bioasq_12b1, df_bioasq_12b2, df_bioasq_12b3, df_bioasq_12b4], ignore_index=True)
print(df_bioasq_test.shape)
df_bioasq_test.head()

In [ ]:
df_bioasq_test.type.value_counts(dropna=False)

In [ ]:
df_bioasq_test["dataset"] = "bioasq"
df_bioasq_test = df_bioasq_test[df_bioasq_test["type"] == "yesno"]
print(df_bioasq_test.exact_answer.value_counts(dropna=False))

In [ ]:
df_bioasq_test = df_bioasq_test.rename(columns={"body": "claim", "snippets": "evidence", "exact_answer": "label"})
df_bioasq_test = df_bioasq_test[["claim", "evidence", "label", "dataset"]]
df_bioasq_test.label = df_bioasq_test.label.apply(lambda x: "supports" if x == "yes" else "refutes")
print(df_bioasq_test.label.value_counts(dropna=False))
df_bioasq_test.head()

In [ ]:
df_bioasq_test['evidence'] = df_bioasq_test['evidence'].apply(parse_evidence)

In [ ]:
df_bioasq_test.head()

##### COVERT FC

In [ ]:
df_covert_fc = pd.read_json(test_mapping['covert_fc'], lines=True)
print(df_covert_fc.shape)
print(df_covert_fc.label.value_counts(dropna=False))
df_covert_fc.head()

In [ ]:
df_covert_fc = df_covert_fc[["claim", "evidence", "label"]]
df_covert_fc["dataset"] = "covert_fc"
df_covert_fc["label"] = df_covert_fc["label"].apply(
    lambda x: "supports" if x == "SUPPORTS" else ("refutes" if x == "REFUTES" else x.lower())
)
print(df_covert_fc.label.value_counts(dropna=False))
df_covert_fc.head()

##### FEVEROUS

In [ ]:
df_feverous_test = pd.read_json(test_mapping['feverous'])
print(df_feverous_test.shape)
print(df_feverous_test.label.value_counts(dropna=False))
df_feverous_test.head()

In [ ]:
df_feverous_test["dataset"] = "feverous"
df_feverous_test = df_feverous_test[["claim", "evidence", "label", "dataset"]]
print(df_feverous_test.label.value_counts(dropna=False))
df_feverous_test.head()

##### HealthFC

In [ ]:
df_healthfc_test_raw = pd.read_csv(test_mapping['healthfc_raw'])
df_healthfc_test_raw.head()

In [ ]:
df_healthfc_test_raw["dataset"] = "healthfc_golden"
df_healthfc_test_raw = df_healthfc_test_raw.rename(columns={"en_claim": "claim", "en_explanation": "evidence"})
df_healthfc_test_raw = df_healthfc_test_raw[["claim", "evidence", "label", "dataset"]]
df_healthfc_test_raw.label = df_healthfc_test_raw.label.apply(
    lambda x: "supports" if x == 0 else ("refutes" if x == 2 else "not enough info")
)
print(df_healthfc_test_raw.label.value_counts(dropna=False))
df_healthfc_test_raw.head()

##### HealthFC - WebSearch Tavily 20

In [ ]:
df_healthfc_test = pd.read_csv(test_mapping['healthfc'])
print(df_healthfc_test.shape)
print(df_healthfc_test.label.value_counts(dropna=False))
df_healthfc_test.head()

In [ ]:
df_healthfc_test["dataset"] = "healthfc_open"
df_healthfc_test = df_healthfc_test.rename(columns={"en_claim": "claim", "evidence_tavily_20_results": "evidence"})
df_healthfc_test = df_healthfc_test[["claim", "evidence", "label", "dataset"]]
df_healthfc_test.label = df_healthfc_test.label.apply(lambda x: "supports" if x == 0 else "refutes")
print(df_healthfc_test.label.value_counts(dropna=False))
df_healthfc_test.head()

##### HealthVer Test

In [ ]:
df_healthver_test = pd.read_json(test_mapping['healthver'])
print(df_healthver_test.shape)
print(df_healthver_test.label.value_counts(dropna=False))
df_healthver_test.head()

In [ ]:
df_healthver_test["dataset"] = "healthver"
df_healthver_test["label"] = df_healthver_test["label"].apply(
    lambda x: "supports" if x == "Supports" else ("refutes" if x == "Refutes" else x.lower())
)

print(df_healthver_test.label.value_counts(dropna=False))
df_healthver_test.head()

##### Hover Test

In [ ]:
df_hover_test = pd.read_json(test_mapping['hover'])
print(df_hover_test.shape)
print(df_hover_test.label.value_counts(dropna=False))
df_hover_test.head()

In [ ]:
df_hover_test["dataset"] = "hover"
df_hover_test = df_hover_test[["claim", "evidence", "label", "dataset"]]
print(df_hover_test.label.value_counts(dropna=False))
df_hover_test.head()

##### LLM Aggrefact

In [ ]:
df_llm_aggrefact_test = pd.read_json(test_mapping['llm_aggrefact'])
print(df_llm_aggrefact_test.shape)
print(df_llm_aggrefact_test.label.value_counts(dropna=False))
df_llm_aggrefact_test.head()

In [ ]:
df_llm_aggrefact_test["dataset"] = "llm_aggrefact"
df_llm_aggrefact_test.label = df_llm_aggrefact_test.label.apply(lambda x: "supports" if x == 1 else "refutes")
print(df_llm_aggrefact_test.label.value_counts(dropna=False))
df_llm_aggrefact_test.head()

##### Feverous Open

In [ ]:
df_feverous_open_test = pd.read_json(test_mapping['feverous_open'])
print(df_feverous_open_test.shape)
print(df_feverous_open_test.label.value_counts(dropna=False))
df_feverous_open_test.head()

In [ ]:
df_feverous_open_test["dataset"] = "feverous_open"
df_feverous_open_test = df_feverous_open_test[["claim", "evidence", "label", "dataset"]]
df_feverous_open_test.head()

##### HealthVer Open

In [ ]:
df_healthver_open_test = pd.read_json(test_mapping['healthver_open'])
print(df_healthver_open_test.shape)
print(df_healthver_open_test.label.value_counts(dropna=False))
df_healthver_open_test.head()

In [ ]:
df_healthver_open_test["dataset"] = "healthver_open"
df_healthver_open_test["label"] = df_healthver_open_test["label"].apply(
    lambda x: "supports" if x == "Supports" else ("refutes" if x == "Refutes" else x.lower())
)

print(df_healthver_open_test.label.value_counts(dropna=False))
df_healthver_open_test.head()

##### Hover Open

In [ ]:
df_hover_open_test = pd.read_json(test_mapping['hover_open'])
print(df_hover_open_test.shape)
print(df_hover_open_test.label.value_counts(dropna=False))
df_hover_open_test.head()

In [ ]:
df_hover_open_test["dataset"] = "hover_open"
df_hover_open_test = df_hover_open_test[["claim", "evidence", "label", "dataset"]]
print(df_hover_open_test.label.value_counts(dropna=False))
df_hover_open_test.head()

##### SciFact Dev

In [ ]:
df_scifact_dev = pd.read_json(test_mapping['scifact_dev'])
print(df_scifact_dev.shape)
print(df_scifact_dev.label.value_counts(dropna=False))
df_scifact_dev.head()

In [ ]:
df_scifact_dev["dataset"] = "scifact_dev"
df_scifact_dev.label = df_scifact_dev.label.apply(lambda x: "supports" if x == "SUPPORT" else ("refutes" if x == "REFUTE" or x == "CONTRADICT" else x.lower()))
print(df_scifact_dev.label.value_counts(dropna=False))
df_scifact_dev.head()

##### SciFact Train

In [ ]:
df_scifact_train = pd.read_json(test_mapping['scifact_train'])
print(df_scifact_train.shape)
print(df_scifact_train.label.value_counts(dropna=False))
df_scifact_train.head()

In [ ]:
df_scifact_train["dataset"] = "scifact_train"
df_scifact_train.label = df_scifact_train.label.apply(lambda x: "supports" if x == "SUPPORT" else ("refutes" if x == "REFUTE" or x == "CONTRADICT" else x.lower()))
print(df_scifact_train.label.value_counts(dropna=False))
df_scifact_train.head()

##### VitaminC Dev

In [ ]:
df_vitaminc_dev = pd.read_json(test_mapping['vitaminc_dev'])
print(df_vitaminc_dev.shape)
print(df_vitaminc_dev.label.value_counts(dropna=False))
df_vitaminc_dev.head()

In [ ]:
df_vitaminc_dev["dataset"] = "vitaminc_dev"
df_vitaminc_dev.label = df_vitaminc_dev.label.apply(lambda x: "supports" if x == "SUPPORTS" else ("refutes" if x == "REFUTES" else x.lower()))
print(df_vitaminc_dev.label.value_counts(dropna=False))
df_vitaminc_dev.head()

##### VitaminC Test

In [ ]:
df_vitaminc_test = pd.read_json(test_mapping['vitaminc_test'])
print(df_vitaminc_test.shape)
print(df_vitaminc_test.label.value_counts(dropna=False))
df_vitaminc_test.head()

In [ ]:
df_vitaminc_test["dataset"] = "vitaminc_test"
df_vitaminc_test.label = df_vitaminc_test.label.apply(lambda x: "supports" if x == "SUPPORTS" else ("refutes" if x == "REFUTES" else x.lower()))
print(df_vitaminc_test.label.value_counts(dropna=False))
df_vitaminc_test.head()

### Merge

In [ ]:
df_train_merged = pd.concat([
    df_pubmedqa,
    df_bioasq,
    df_pubmedqa_artificial,
    df_covid_fact,
    df_feverous,
    df_healthver,
    df_hover,
]).reset_index(drop=True)

df_test_merged = pd.concat([
    df_bioasq_test,
    df_covert_fc,
    df_feverous_test,
    df_healthfc_test_raw,
    df_healthfc_test,
    df_healthver_test,
    df_hover_test,
    df_llm_aggrefact_test,
    df_feverous_open_test,
    df_healthver_open_test,
    df_hover_open_test,
    df_scifact_dev,
    df_scifact_train,
    df_vitaminc_dev,
    df_vitaminc_test,
]).reset_index(drop=True)

In [ ]:
print(df_train_merged.shape)
print(df_train_merged.label.value_counts(dropna=False))
print(df_train_merged.dataset.value_counts(dropna=False))
df_train_merged.head()

In [ ]:
print(df_test_merged.shape)
print(df_test_merged.label.value_counts(dropna=False))
print(df_test_merged.dataset.value_counts(dropna=False))
df_test_merged.head()

### Decontamination

In [ ]:
# Adapted from simplescaling/s1 (data/decontaminate_util.py), Apache License 2.0:
# https://github.com/simplescaling/s1
import collections
import numpy as np
from tqdm import tqdm


def normalize_string(text):
    """Basic string normalization."""
    # Convert to lowercase and normalize whitespace
    text = text.lower().strip()
    # Replace multiple spaces with single space
    text = ' '.join(text.split())
    return text

def word_ngrams(text, n):
    """Generate word-level n-grams from text."""
    words = text.split()
    return [' '.join(words[i:i+n]) for i in range(len(words) - n + 1)]

def build_ngram_lookup(documents, ngram_size=13):
    """Build ngram lookup for documents."""
    print(f"Building {ngram_size}-gram lookup...")
    lookup = collections.defaultdict(set)

    for doc_id, document in enumerate(tqdm(documents)):
        normalized_text = normalize_string(document)
        ngrams = word_ngrams(normalized_text, ngram_size)
        for ngram in ngrams:
            lookup[ngram].add(doc_id)
    
    return lookup


def find_contaminated_questions(test_lookup, train_lookup):
    """Find overlapping documents based on ngram matches."""
    contaminated_ids = set()
    matched_ngrams = []
    
    for ngram, train_doc_ids in tqdm(train_lookup.items(), desc="Checking overlaps"):
        if ngram in test_lookup:
            contaminated_ids.update(train_doc_ids)
            matched_ngrams.append(ngram)
    
    # Print some example matches for inspection
    if matched_ngrams:
        print("\nExample matching n-grams:")
        for ngram in matched_ngrams[:5]:  # Show first 5 matches
            print(f"  - {ngram}")
    
    return contaminated_ids

In [ ]:
def analyze_text_statistics(texts, text_type="Text"):
    """Calculate word count statistics for a list of texts."""
    # Handle different data types - convert everything to string first, then filter
    valid_texts = []
    for text in texts:
        # Convert to string first
        text_str = str(text)
        # Then check if it's meaningful (not just 'nan' or empty)
        if text_str.lower() not in ['nan', 'none', ''] and len(text_str.strip()) > 0:
            valid_texts.append(text_str)
    
    word_counts = [len(text.split()) for text in valid_texts]
    
    if not word_counts:
        print(f"No valid {text_type.lower()} found!")
        return []
    
    print(f"\n{text_type} Statistics:")
    print(f"Count: {len(word_counts)}")
    print(f"Average: {np.mean(word_counts):.2f}")
    print(f"Min: {np.min(word_counts)}")
    print(f"Max: {np.max(word_counts)}")
    print(f"Median: {np.median(word_counts):.2f}")
    print(f"25th percentile: {np.percentile(word_counts, 25):.2f}")
    print(f"75th percentile: {np.percentile(word_counts, 75):.2f}")
    print(f"95th percentile: {np.percentile(word_counts, 95):.2f}")
    
    return word_counts

### Text statistics

Word count statistics of claims and evidence in the training and held-out sets, used to choose the n-gram sizes for decontamination.

In [ ]:
print("=" * 50)
print("TRAINING DATA ANALYSIS")
print("=" * 50)

# Analyze training claims
train_claim_counts = analyze_text_statistics(df_train_merged['claim'], "Training Claims")

# Analyze training evidence
train_evidence_counts = analyze_text_statistics(df_train_merged['evidence'], "Training Evidence")

In [ ]:
print("\n" + "=" * 50)
print("TEST DATA ANALYSIS")
print("=" * 50)

# Analyze test claims
test_claim_counts = analyze_text_statistics(df_test_merged['claim'], "Test Claims")

# Analyze test evidence
test_evidence_counts = analyze_text_statistics(df_test_merged['evidence'], "Test Evidence")

### Decontamination

Training instances that share a word n-gram with any held-out set are removed: 8-grams on claims, 10-grams on evidence.

Claim

In [ ]:
train_claims = [str(text) for text in df_train_merged['claim'] if pd.notna(text)]
test_claims = [str(text) for text in df_test_merged['claim'] if pd.notna(text)]

train_lookup = build_ngram_lookup(train_claims, 8)
test_lookup = build_ngram_lookup(test_claims, 8)
contaminated_ids = find_contaminated_questions(test_lookup, train_lookup)

print(f"Contaminated training samples (claims): {len(contaminated_ids)} / {len(train_claims)}")
print(f"Training contamination rate (claims): {len(contaminated_ids)/len(train_claims)*100:.1f}%")

Evidence

In [ ]:
train_evidence = [str(text) for text in df_train_merged['evidence']]
test_evidence = [str(text) for text in df_test_merged['evidence']]

train_evidence_lookup = build_ngram_lookup(train_evidence, 10)
test_evidence_lookup = build_ngram_lookup(test_evidence, 10)
contaminated_evidence_ids = find_contaminated_questions(test_evidence_lookup, train_evidence_lookup)

print(f"Contaminated training samples (evidence): {len(contaminated_evidence_ids)} / {len(train_evidence)}")
print(f"Training contamination rate (evidence): {len(contaminated_evidence_ids)/len(train_evidence)*100:.1f}%")

In [ ]:
all_train_contaminated = contaminated_ids.union(contaminated_evidence_ids)

print(f"Contaminated training samples to remove: {len(all_train_contaminated)} / {len(df_train_merged)}")
print(f"Training contamination rate: {len(all_train_contaminated)/len(df_train_merged)*100:.1f}%")

# Remove contaminated samples from training set only (test set remains unchanged)
df_train_merged = df_train_merged.drop(df_train_merged.index[list(all_train_contaminated)]).reset_index(drop=True)

print(f"Clean datasets: Test={len(df_test_merged)} (unchanged), Train={len(df_train_merged)} (removed {len(all_train_contaminated)})")

#### Prepare dataset

In [ ]:
df_train_merged.head()

In [ ]:
df_train_merged.label.value_counts(dropna=False)

In [ ]:
df_train_merged.dataset.value_counts(dropna=False)

## Final training set sampling

Only `supports` and `refutes` instances are kept. Each dataset is sampled with a 50/50 label split up to a quota, limited by the number of instances available per label after decontamination:

| Dataset | Quota | Sampled |
|---|---|---|
| BioASQ | 1,000 | 851 |
| PubMedQA (labeled) | 993 | 831 |
| PubMedQA (artificial) | 900 | 900 |
| FEVEROUS | 250 | 250 |
| HoVer | 250 | 250 |
| COVID-Fact | 250 | 250 |
| HealthVer | 250 | 250 |

**Total**: 3,582 instances.

In [ ]:
df_train_filtered = df_train_merged[df_train_merged['label'].isin(['supports', 'refutes'])].copy()
print(f"Total samples after filtering for supports/refutes: {len(df_train_filtered)}")
print("\nLabel distribution:")
print(df_train_filtered['label'].value_counts())
print("\nDataset distribution:")
print(df_train_filtered['dataset'].value_counts())

In [ ]:
# Define sampling quotas per dataset
dataset_quotas = {
    'bioasq': 1000,
    'pubmedqa': 993,
    'pubmedqa_artificial': 900,
    'feverous': 250,
    'hover': 250,
    'covidfact': 250,
    'healthver': 250
}

def sample_balanced_dataset(df, n_samples, random_state=42):
    """Sample exactly n_samples with 50/50 label split."""
    supports = df[df['label'] == 'supports']
    refutes = df[df['label'] == 'refutes']
    
    n_per_label = n_samples // 2
    
    supports_sample = supports.sample(n=min(n_per_label, len(supports)), random_state=random_state)
    refutes_sample = refutes.sample(n=min(n_per_label, len(refutes)), random_state=random_state)
    
    return pd.concat([supports_sample, refutes_sample], ignore_index=True)

sampled_dfs = []

for dataset_name, quota in dataset_quotas.items():
    dataset_df = df_train_filtered[df_train_filtered['dataset'] == dataset_name]
    sampled = sample_balanced_dataset(dataset_df, quota)
    print(f"{dataset_name.capitalize()} - Sampled {len(sampled)} with labels: {sampled['label'].value_counts().to_dict()}")
    
    sampled_dfs.append(sampled)

# Combine all sampled datasets
df_train_sample = pd.concat(sampled_dfs, ignore_index=True)

print(f"\nFinal training set size: {len(df_train_sample)}")
print("\nFinal label distribution:")
print(df_train_sample['label'].value_counts())
print("\nFinal dataset distribution:")
print(df_train_sample['dataset'].value_counts())

In [ ]:
# Display sample of the training data
df_train_sample.head()

In [ ]:
# Save the sampled dataset
os.makedirs(OUTPUT_DIR, exist_ok=True)

sample_path = os.path.join(OUTPUT_DIR, "sample.jsonl")
df_train_sample.to_json(sample_path, orient="records", lines=True)
print(f"Sample saved to: {sample_path}")